In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Load variables
constants = pd.read_csv('constants.csv', delimiter = ';', header = None, names=['variable', 'value'], index_col='variable')['value'].to_dict()
energy_density = constants['energy_density']

In [ ]:
### LOAD FUNCTIONS ###

def load_emission_factors(
    path, 
    sheet_fossil = "Emmission_Factors_Fossils",
    sheet_grids = "Emmission_Factors_Power"
):
    """
    Load emission intensity data for fossil fuels and grid electricity.

    Parameters
    ----------
    path : str or Path
        Path to emission_factors_fossil_grid.xlsx.
    sheet_fossil : str
        Sheet with fossil-fuel CO2 intensities (upper/lower range per product) in kg CO2-eq / unit of product.
    sheet_power : str
        Sheet with grid electricity CO2 intensities in g CO2-eq / kWh.
        (rows = years, columns = zones / regions).

    Returns
    -------
    df_fossil_ef : pd.DataFrame
        Columns = ['Upper range', 'Lower range'], index = product names.
        Unit: kg CO2 / unit product.
    df_grid_ef : pd.DataFrame
        Index = year (int), columns = zones.
        Unit: g CO2 / kWh
    """

    df_fossil_ef = pd.read_excel(
        path, 
        sheet_name=sheet_fossil, 
        index_col=0,
        usecols = "A:C"
    )

    df_grid_ef = pd.read_excel(
        path, 
        sheet_name=sheet_grids, 
        index_col=0
    )

    return df_fossil_ef, df_grid_ef



def load_cost_factors(
    path,
    sheet_fossil = "Prices_Fossil_Alternatives",
    sheet_eua = "EUA_Prices",
):
    """
    Load fossil-fuel market prices and EUA (EU Allowance) prices from Excel.

    Parameters
    ----------
    path : str or Path
        Path to cost_factors_fossil_alternatives.xlsx
    sheet_fossil : str
        Sheet name with fossil prices (rows = years, cols = products).
    sheet_eua : str
        Sheet name with EUA prices (rows = years, col 'EUA').

    Returns
    -------
    fossil_prices : pd.DataFrame
        Index = year (int), columns = product names, values in € / unit product.
    eua_prices : dict
        Keys = year, values = prices in € / t CO2-eq.
    """

    fossil_prices = pd.read_excel(
        path, 
        sheet_name=sheet_fossil, 
        index_col=0,
        usecols = "A:H"
    )

    eua_prices = pd.read_excel(
        path, 
        sheet_name=sheet_eua, 
        index_col=0,
        usecols = "A:B"
    )['EUA'].to_dict()

    return fossil_prices, eua_prices

In [15]:
df_fossil_ef, df_grid_ef = load_emission_factors("/Users/marianapessoa/Library/CloudStorage/OneDrive-CBS-CopenhagenBusinessSchool/Documents/AutoCodingTask/emission_factors_fossil_grid.xlsx")

In [17]:
df_fossil_ef.loc["Methanol", "Upper range"]

np.float64(2.863)

In [ ]:
### HELPER FUNCTIONS ###

def safe_div(
        numer, 
        denom
    ):
    """
    Makes a safe division: 
    1) returns None if an error occurs, or numer or denom are None
    2) returns None if denom is 0
    
    """
    if numer is None or denom is None:
        return None
    try:
        denom_f = float(denom)
        if denom_f == 0:
            return None
        return numer / denom_f
    except Exception:
        return None
    

def gCO2_per_kwh_to_kgCO2_per_wh(value):
    """Convert g CO2 / kWh  →  kg CO2 / Wh  (divide by 1e6)."""
    if value is None:
        return None
    return float(value) / 1e6

def mwh_to_wh(value):
    """Convert MWh → Wh (multiply by 1e6)."""
    if value is None:
        return None
    return float(value) * 1e6

def kg_to_t(value):
    """Convert kg → t (devide by 1e3)."""
    if value is None:
        return None
    return float(value) / 1e3

In [ ]:
def compute_electricity_breakdown(scenario):
    """
    Calculate the electricity breakdown. 
    Electricity can come from on-site or PPA RES (wind or PV), and the grid (from more than one country).


    Parameters
    ----------
    Expected keys in `scenario`:
        (option 1)
        production_pv_mwh               : float  — on-site PV generation (MWh/a)
        production_wind_onshore_mwh     : float  — on-site onshore generation (MWh/a)
        production_wind_offshore_mwh    : float  — on-site offshore generation (MWh/a)
        el_from_grid_mwh                : float  — electricity imported from the grid (MWh/a)
        el_to_grid_mwh                  : float  — electricity exported to the grid (MWh/a)

        (option 2)
        pv_ppa_mwh                      : float  — electricity from PV PPA (MWh/a), default 0
        wind_ppa_mwh                    : float  — electricity from wind PPA (MWh/a), default 0
        grid_from_mwh                   : dict   — {zone_name: MWh, ...} multi-zone grid import

    

    Returns
    --------
    Returns dict with keys:
        el_used_total_mwh               : total electricity consumed in production (MWh/a)
        res_used_mwh                    : total on-site RES electricity used for production (MWh/a)
        pv_ppa_mwh                      : PV PPA electricity (MWh/a)
        wind_ppa_mwh                    : wind PPA electricity (MWh/a)
        ppa_total_mwh                   : total PPA electricity (MWh/a)
        grid_total_mwh                  : total grid electricity used in production (MWh/a)
        grid_zone_breakdown_mwh         : {zone: MWh, ...}

    """
    pv_produced = float(scenario.get("production_pv_mwh") or 0.0)
    wind_on_produced = float(scenario.get("production_wind_onshore_mwh") or 0.0)
    wind_off_produced = float(scenario.get("production_wind_offshore_mwh") or 0.0)
    el_from_grid = float(scenario.get("el_from_grid_mwh") or 0.0)
    el_to_grid = float(scenario.get("el_to_grid_mwh") or 0.0)

    wind_ppa = float(scenario.get("wind_ppa_mwh") or 0.0)
    pv_ppa = float(scenario.get("pv_ppa_mwh") or 0.0)
    ppa_total = wind_ppa + pv_ppa


    # If 'grid_from' is provided, then there isn't a need to calculate the amount of electricity from the grid
    grid_zone = scenario.get("grid_from_mwh") or {}

    if isinstance(grid_zone, dict) and grid_zone:

        grid_total = sum(v for v in grid_zone.values() if v is not None)
        res_used = 0
        
    else: # Single-source grid:
        grid_zone = {}
        
        res_produced = pv_produced + wind_on_produced + wind_off_produced
        el_used = res_produced + el_from_grid - el_to_grid
        res_used = res_produced - el_to_grid
        res_used_rel = safe_div(res_used, el_used) if el_used > 0 else None
        
        grid_total = el_from_grid * (1.0 - (res_used_rel or 0.0))


    el_used = res_used + ppa_total + grid_total

    return {
        "el_used_total_mwh": el_used,
        "res_used_mwh": res_used,
        "pv_ppa_mwh": pv_ppa,
        "wind_ppa_mwh": wind_ppa,
        "ppa_total_mwh": ppa_total,
        "grid_total_mwh": grid_total,
        "grid_zone_breakdown": grid_zone,
    }

In [ ]:
def compute_hub_emissions(
        el_breakdown,
        df_grid_ef,
        year,
        ppa_emission_factor_gkwh,
        primary_zone = "DK1",
):
    """
    Compute the CO2 emissions from hub's electricity (grid) consumption.

    Parameters
    ----------
    el_breakdown                : dict from compute_electricity_breakdown()
    df_grid_ef                  : pandas.DataFrame with grid emission factors (g CO2/kWh), index=year, columns=zone
    year                        : int — the year for which to look up the grid emission factor
    ppa_emission_factor_gkwh    : float — CO2 intensity of PPA electricity (g CO2/kWh)
    primary_zone                : str — which column of df_grid_ef to use when no zone breakdown

    Returns dict with keys
    ----------------------
    hub_emissions_tco2        : total annual hub CO2 emissions (t CO2/a)
    emissions_by_source       : {source: tCO2, ...}
    """

    # Calculate emissions from grid
    sources: dict[str, float | None] = {}

    zone_breakdown = el_breakdown.get("grid_zone_breakdown") or {}
    if zone_breakdown:
        for zone, mwh in zone_breakdown.items():
            # look up emission factor for the zone (column) and year
            ef_zone = None
            if zone in df_grid_ef.columns and year in df_grid_ef.index:
                ef_zone = gCO2_per_kwh_to_kgCO2_per_wh(df_grid_ef.at[year, zone])
            sources[f"grid_{zone}"] = kg_to_t(mwh_to_wh(mwh) * ef_zone)
    else:
        grid_mwh = el_breakdown.get("grid_total_mwh") or 0.0
        ef_primary = None
        if primary_zone in df_grid_ef.columns and year in df_grid_ef.index:
            ef_primary = gCO2_per_kwh_to_kgCO2_per_wh(df_grid_ef.at[year, primary_zone])
        sources[f"grid_{primary_zone}"] = kg_to_t(mwh_to_wh(grid_mwh) * ef_primary)


    # Calculate emissions from PPA
    ef_ppa = gCO2_per_kwh_to_kgCO2_per_wh(ppa_emission_factor_gkwh)


    # PPA sources (wind + PV via PPA)
    wind_ppa_mwh = el_breakdown.get("wind_ppa_mwh")
    sources["wind_ppa"] = kg_to_t(mwh_to_wh(wind_ppa_mwh) * ef_ppa)
    pv_ppa_mwh = el_breakdown.get("pv_ppa_mwh")
    sources["pv_ppa"] = kg_to_t(mwh_to_wh(wind_ppa_mwh) * ef_ppa)

    

    total = sum(v for v in sources.values() if v is not None) 

    return {
        "hub_emissions_tco2": total,
        "emissions_by_source": sources,
    }